Fase 2 Modul 3: Feature Engineering + Cleaning Log Final

In [1]:
from pathlib import Path
import pandas as pd

p = Path("data/dirty_tips2.csv") if Path("data/dirty_tips.csv").exists() else Path("../data/dirty_tips2.csv")
# sederhanakan: pakai pola biasa yang sudah kamu kuasai
p = Path("data/dirty_tips2.csv") if Path("data/dirty_tips2.csv").exists() else Path("../data/dirty_tips2.csv")
df = pd.read_csv(p)
# anggap df sudah lewat clean Modul 1-2: sex/day standar, tanggal parse
# di sini langsung bikin fitur:
df["tip_pct"] = df["tip"] / df["total_bill"] * 100  # royalitas %, fitur No.1 analis resto
df["total_bayar"] = df["total_bill"] + df["tip"]  # uang keluar beneran
df["harga_per_orang"] = df["total_bill"] / df["size"]  # mahal tidaknya per kepala
df["is_weekend"] = df["day"].isin(["Sat", "Sun"])  # boolean flag, gampang di-groupby
# kategori bill pakai kuantil (biar seimbang, bukan tebak angka):
df["bill_level"] = pd.qcut(df["total_bill"], q=3, labels=["hemat", "sedang", "mahal"])
print(df[["total_bill","tip_pct","total_bayar","harga_per_orang","is_weekend","bill_level"]].head(3))

   total_bill    tip_pct  total_bayar  harga_per_orang  is_weekend bill_level
0      500.00   0.202000       501.01       250.000000       False      mahal
1       10.34 -48.355899         5.34         3.446667       False      hemat
2       21.01  16.658734        24.51         7.003333       False     sedang


In [2]:
# Pipeline final + log yang bisa diaudit

log = [f"Awal: {df.shape}"]
# contoh isi log final (gabung Modul 1+2+3):
# 1. drop 3 duplikat, 2. tip median, day modus, 3. flag 9 outlier (tidak drop)
# 4. tanggal 64 rusak -> NaT + flag, 5. sex 5 + day 4 dirapikan
# 6. tambah 5 fitur di atas
# Aturan log bagus: tiap baris = APA + BERAPA + KENAPA
log.append("tip NaN=1 diisi median 2.90 karena distribusi miring kanan")
log.append("outlier 9 di-flag tidak di-drop karena bisa transaksi sah")
# simpan bersih lokal (tidak ke-push, sengaja):
out = Path("data/clean_tips.csv") if Path("data").exists() else Path("../data/clean_tips.csv")
df.to_csv(out, index=False)
print(f"simpan bersih: {out} {df.shape}")
print("\n".join(log))

simpan bersih: ..\data\clean_tips.csv (244, 13)
Awal: (244, 13)
tip NaN=1 diisi median 2.90 karena distribusi miring kanan
outlier 9 di-flag tidak di-drop karena bisa transaksi sah
